# Setup ENCODE data
## Setup log normalized TPM data of lncRNA-TF pairs
### Author: Martin Loza
### Date: 25/12/16

Now, let's select expression data of selected gene pairs

In [27]:
# Change R language to English
Sys.setenv(LANGUAGE = "en")

# Init
suppressPackageStartupMessages({
    library(dplyr)
    library(stringr)
    library(ggplot2)
    library(patchwork)
})

# Local variables 
seed = 777
date = "251216"

# Define colors for strand plots
red = "#E41A1C"
blue = "#090a0bff"
# Define colors for gene types
green = "#4DAF4A"
purple = "#984EA3"
text_size = 18
width = 18.6
dot_size = 4
line_size = 1.5
dpi = 300

data_dir = "/Users/martin/Documents/Projects/lncRNA_TF_pairs_analysis/Data/ENCODE/merged/"
out_dir = "/Users/martin/Documents/Projects/lncRNA_TF_pairs_analysis/Data/ENCODE/selected_gene_pairs/"

# Local Functions


### Load and setup the data

In [4]:
# load the merged normalized data
merged_normalized <- read.table(paste0(data_dir, "merged_log_normalized_tpm_251216.tsv"), 
                        header = TRUE, sep = "\t", stringsAsFactors = FALSE)
head(merged_normalized)[1:5]

,gene_id,log_TPM_ENCSR000CUE,log_TPM_ENCSR000AHH,log_TPM_ENCSR000AAQ,log_TPM_ENCSR000AEU
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
1,ENSG00000000003,1.6882491,1.3454724,2.83556352,1.8373700
2,ENSG00000000005,0.0000000,0.0000000,0.00000000,0.0000000
3,ENSG00000000419,2.1317968,2.9407480,2.82967769,1.7173951
4,ENSG00000000457,0.7371641,1.3244190,1.12492960,0.9001613
5,ENSG00000000460,0.7030975,0.9122827,0.59332685,0.4700036
6,ENSG00000000938,0.0000000,0.5306283,0.01980263,1.2296406


Load the metadata

In [3]:
# load the metadata
metadata <- read.table("/Users/martin/Documents/Projects/lncRNA_TF_pairs_analysis/Data/Supplementary/Supplementary_Table_ENCODE_datasets_251216.tsv", 
                      header = TRUE, sep = "\t", stringsAsFactors = FALSE, 
                      quote = "", comment.char = "")
head(metadata)

,Accession,RNA_seq_file,Assay.name,Assay.title,Biosample.classification,Biosample.term.name,Dbxrefs,Organism,Life.stage,simple_label,unique_sample_id
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,ENCSR000CUE,ENCFF633DRY,RNA-seq,total RNA-seq,primary cell,articular chondrocyte of knee joint,"GEO-obsolete:GSM984611,UCSC-ENCODE-hg19:wgEncodeEH002674,GEO:GSE78607",Homo sapiens,adult,chondrocytes,chondrocytes_adult
2,ENCSR000AHH,ENCFF549QYX,RNA-seq,total RNA-seq,tissue,heart,GEO:GSE87943,Homo sapiens,adult,heart,heart_adult
3,ENCSR000AAQ,ENCFF733XOH,RNA-seq,total RNA-seq,primary cell,renal cortical epithelial cell,GEO:GSE78544,Homo sapiens,adult,renal_epithelial,renal_epithelial_adult
4,ENCSR000AEU,ENCFF960OCQ,RNA-seq,total RNA-seq,tissue,liver,GEO:GSE78562,Homo sapiens,adult,liver,liver_adult
5,ENCSR000CUF,ENCFF220VSM,RNA-seq,total RNA-seq,primary cell,osteoblast,"UCSC-ENCODE-hg19:wgEncodeEH002675,GEO-obsolete:GSM984610,GEO:GSE78608",Homo sapiens,adult,osteoblasts,osteoblasts_adult
6,ENCSR369RVN,ENCFF646YNM,RNA-seq,total RNA-seq,primary cell,cardiac ventricle fibroblast,GEO:GSE78645,Homo sapiens,adult,cardiac_fibroblasts,cardiac_fibroblasts_adult


In [6]:
cat("Any duplicated Biosample.term.name	in metadata?: ", any(duplicated(metadata$Biosample.term.name)), "\n")
cat("Any duplicated unique_sample_id in metadata?: ", any(duplicated(metadata$unique_sample_id)), "\n")

Any duplicated Biosample.term.name	in metadata?:  TRUE 
Any duplicated unique_sample_id in metadata?:  FALSE 


In [7]:
# Extract sample names from column names (remove "log_TPM_" prefix)
sample_names <- colnames(merged_normalized)[-1]  # Exclude gene_id column
sample_names <- gsub("log_TPM_", "", sample_names)

cat("Number of samples:", length(sample_names), "\n")
head(sample_names)

Number of samples: 20 


[1] "ENCSR000CUE" "ENCSR000AHH" "ENCSR000AAQ" "ENCSR000AEU" "ENCSR000CUF"
[6] "ENCSR369RVN"

In [8]:
# Create a batch vector based on the experimental accession
# Each sample corresponds to one experiment (ENCSR ID)
# We'll match the sample names with the Accession column in metadata

batch_info <- data.frame(
  sample = sample_names,
  stringsAsFactors = FALSE
)

# Match with metadata to get tissue/cell type and other info
batch_info$tissue <- sapply(sample_names, function(s) {
  idx <- which(metadata$Accession == s)
  if(length(idx) > 0) {
    return(metadata$Biosample.term.name[idx[1]])
  } else {
    return(NA)
  }
})

batch_info$biosample_class <- sapply(sample_names, function(s) {
  idx <- which(metadata$Accession == s)
  if(length(idx) > 0) {
    return(metadata$Biosample.classification[idx[1]])
  } else {
    return(NA)
  }
})

batch_info$simple_label <- sapply(sample_names, function(s) {
  idx <- which(metadata$Accession == s)
  if(length(idx) > 0) {
    return(metadata$simple_label[idx[1]])
  } else {
    return(NA)
  }
})

batch_info$unique_sample_id <- sapply(sample_names, function(s) {
  idx <- which(metadata$Accession == s)
  if(length(idx) > 0) {
    return(metadata$unique_sample_id[idx[1]])
  } else {
    return(NA)
  }
})

head(batch_info)
cat("\nNumber of (samples):", length(unique(batch_info$sample)), "\n")

,sample,tissue,biosample_class,simple_label,unique_sample_id
,<chr>,<chr>,<chr>,<chr>,<chr>
1,ENCSR000CUE,articular chondrocyte of knee joint,primary cell,chondrocytes,chondrocytes_adult
2,ENCSR000AHH,heart,tissue,heart,heart_adult
3,ENCSR000AAQ,renal cortical epithelial cell,primary cell,renal_epithelial,renal_epithelial_adult
4,ENCSR000AEU,liver,tissue,liver,liver_adult
5,ENCSR000CUF,osteoblast,primary cell,osteoblasts,osteoblasts_adult
6,ENCSR369RVN,cardiac ventricle fibroblast,primary cell,cardiac_fibroblasts,cardiac_fibroblasts_adult



Number of (samples): 20 


Load the lncRNA-TF gene pairs

In [11]:
gene_pairs_file <- "/Users/martin/Documents/Projects/lncRNA_TF_pairs_analysis/Data/Annotated_ncRNA_PCG_pairs/human_unique_lncRNA_TF_pairs_10000bp_251215.tsv"
# Load the unique gene pairs data
gene_pairs <- read.table(gene_pairs_file, sep = "\t", header = TRUE, comment.char = "", fill = TRUE, row.names = NULL)
dim(gene_pairs)
head(gene_pairs,2)

[1] 1978   18

,chromosome,ncRNA_id,ncrna_tss,ncrna_gene_name,ncrna_strand,gene_biotype,pcg_id,pcg_gene_name,pcg_tss,dna_distance,strand_distance,Family,is_TF,abs_strand_distance,ncrna_gene_id,pcg_gene_id,gene_pair_id,gene_name_pair_id
,<chr>,<chr>,<int>,<chr>,<int>,<chr>,<chr>,<chr>,<int>,<int>,<int>,<chr>,<lgl>,<int>,<chr>,<chr>,<chr>,<chr>
1,11,ENST00000381363,2140644,IGF2-AS,1,lncRNA,ENST00000643349,unnamed,2149603,8959,8959,ZBTB,TRUE,8959,ENSG00000099869,ENSG00000284779,ENSG00000099869_ENSG00000284779,IGF2-AS_unnamed
2,11,ENST00000833483,61756482,MYRF-AS1,-1,lncRNA,ENST00000265460,MYRF,61755389,-1093,1093,NDT80_PhoG,TRUE,1093,ENSG00000124915,ENSG00000124920,ENSG00000124915_ENSG00000124920,MYRF-AS1_MYRF


In [13]:
# get unique lncRNA and TF gene IDs
unique_gene_pairs_ids <- unique(c(gene_pairs$ncrna_gene_id, gene_pairs$pcg_gene_id))
# get unique gene_ids in the ENCODE data. 
unique_encode_gene_ids <- unique(merged_normalized$gene_id)

cat("Number of unique lncRNA and TF gene IDs: ", length(unique_gene_pairs_ids), "\n")
cat("Number of unique lncRNA and TF gene IDs in gene pairs: ", sum(unique_gene_pairs_ids %in% unique_encode_gene_ids), "\n")

Number of unique lncRNA and TF gene IDs:  3074 
Number of unique lncRNA and TF gene IDs in gene pairs:  2092 


We don't have expression data of all genes...

 Let's then select only expression data of related gene pairs

In [17]:
# Select only expression data of genes present in the gene pairs
merged_normalized_selected <- merged_normalized %>%
    filter(gene_id %in% unique_gene_pairs_ids)
cat("Number of genes in merged normalized data after selection: ", nrow(merged_normalized_selected), "\n")

Number of genes in merged normalized data after selection:  2092 


In [18]:
head(merged_normalized_selected)

,gene_id,log_TPM_ENCSR000CUE,log_TPM_ENCSR000AHH,log_TPM_ENCSR000AAQ,log_TPM_ENCSR000AEU,log_TPM_ENCSR000CUF,log_TPM_ENCSR369RVN,log_TPM_ENCSR042GYH,log_TPM_ENCSR146LBD,log_TPM_ENCSR000AAO,⋯,log_TPM_ENCSR719PXC,log_TPM_ENCSR071ZLM,log_TPM_ENCSR000AFL,log_TPM_ENCSR000AEY,log_TPM_ENCSR000AFH,log_TPM_ENCSR000AFO,log_TPM_ENCSR000AFI,log_TPM_ENCSR000AFC,log_TPM_ENCSR000AFB,log_TPM_ENCSR373BDG
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,ENSG00000004848,0.000000000,0.00000000,0.00000000,0.00000000,0.0000000,0.000000000,1.12492960,0.0000000,0.0000000,⋯,0.0000000,0.0000000,0.0295588,0.139761942,0.0000000,0.095310180,0.009950331,0.01980263,0.00000000,0.0000000
2,ENSG00000005073,1.011600912,0.00000000,0.73716407,0.00000000,0.3506569,0.000000000,0.01980263,0.3364722,0.0000000,⋯,0.0000000,1.2470323,0.0000000,0.000000000,0.1988509,0.009950331,0.000000000,0.00000000,0.00000000,1.6370531
3,ENSG00000005102,0.009950331,0.92821930,0.00000000,0.00000000,0.0000000,0.009950331,0.08617770,0.1739533,0.0000000,⋯,0.9820785,0.2390169,1.0079579,0.009950331,0.2070142,0.887891257,0.246860078,0.34358970,0.06765865,0.4824261
4,ENSG00000005436,1.124929597,1.64093658,1.36863943,1.15688120,0.4317824,1.128171091,1.26694760,0.8960880,0.3435897,⋯,1.3635374,0.6830968,1.4398351,0.392042088,1.4350845,2.111424588,1.217875709,2.81959158,1.49514877,1.7491999
5,ENSG00000005513,0.254642218,0.04879016,0.03922071,0.01980263,0.0000000,0.198850859,0.01980263,0.0295588,0.0000000,⋯,1.0224509,0.0000000,1.8594181,0.444685821,2.0095554,2.452727751,0.165514438,0.87129337,0.06765865,0.1570037
6,ENSG00000005801,0.887891257,1.60140574,2.18154676,0.90421815,0.4252677,1.141033005,1.18172720,0.4252677,0.7030975,⋯,0.1133287,0.3576744,0.9593502,0.751416089,1.5933085,2.467251715,0.940007258,2.43449016,1.14103300,2.9101744


For simplicity, let's change the column names to use the unique_sample_id 

In [19]:
# Let's change the column names to match the unique_sample_id in the metadata
colnames(merged_normalized_selected)[-1] <- sapply(colnames(merged_normalized_selected)[-1], function(colname) {
    sample_id <- gsub("log_TPM_", "", colname)
    idx <- which(batch_info$sample == sample_id)
    if(length(idx) > 0) {
        return(batch_info$unique_sample_id[idx[1]])
    } else {
        return(NA)
    }
})
head(merged_normalized_selected)

,gene_id,chondrocytes_adult,heart_adult,renal_epithelial_adult,liver_adult,osteoblasts_adult,cardiac_fibroblasts_adult,ovary_adult,vagina_adult,lung_fibroblasts_adult,⋯,aorta_adult,uterus_adult,tongue_embryonic,frontal_cortex_embryonic,spinal_cord_embryonic,eye_embryonic,stomach_embryonic,lung_embryonic,liver_embryonic,kidney_epithelial_embryonic
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,ENSG00000004848,0.000000000,0.00000000,0.00000000,0.00000000,0.0000000,0.000000000,1.12492960,0.0000000,0.0000000,⋯,0.0000000,0.0000000,0.0295588,0.139761942,0.0000000,0.095310180,0.009950331,0.01980263,0.00000000,0.0000000
2,ENSG00000005073,1.011600912,0.00000000,0.73716407,0.00000000,0.3506569,0.000000000,0.01980263,0.3364722,0.0000000,⋯,0.0000000,1.2470323,0.0000000,0.000000000,0.1988509,0.009950331,0.000000000,0.00000000,0.00000000,1.6370531
3,ENSG00000005102,0.009950331,0.92821930,0.00000000,0.00000000,0.0000000,0.009950331,0.08617770,0.1739533,0.0000000,⋯,0.9820785,0.2390169,1.0079579,0.009950331,0.2070142,0.887891257,0.246860078,0.34358970,0.06765865,0.4824261
4,ENSG00000005436,1.124929597,1.64093658,1.36863943,1.15688120,0.4317824,1.128171091,1.26694760,0.8960880,0.3435897,⋯,1.3635374,0.6830968,1.4398351,0.392042088,1.4350845,2.111424588,1.217875709,2.81959158,1.49514877,1.7491999
5,ENSG00000005513,0.254642218,0.04879016,0.03922071,0.01980263,0.0000000,0.198850859,0.01980263,0.0295588,0.0000000,⋯,1.0224509,0.0000000,1.8594181,0.444685821,2.0095554,2.452727751,0.165514438,0.87129337,0.06765865,0.1570037
6,ENSG00000005801,0.887891257,1.60140574,2.18154676,0.90421815,0.4252677,1.141033005,1.18172720,0.4252677,0.7030975,⋯,0.1133287,0.3576744,0.9593502,0.751416089,1.5933085,2.467251715,0.940007258,2.43449016,1.14103300,2.9101744


In [22]:
cat("Final dimensions of selected merged normalized data: ", dim(merged_normalized_selected), "\n")

Final dimensions of selected merged normalized data:  2092 21 


In [28]:
# Save the merged data
out_file <- file.path(out_dir, paste0("log_normalized_tpm_selected_lncRNA_TF_genes_", date, ".tsv"))
write.table(merged_normalized_selected, file = out_file, sep = "\t", quote = FALSE, row.names = FALSE)